In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeRegressor

In [2]:
print("Loading dataset...")
data = load_breast_cancer()
X, y = data.data, data.target
print(f"Dataset: {data.DESCR.split('==')[0].strip()}")
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

Loading dataset...
Dataset: .. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field

In [3]:
class CustomGradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        
    def fit(self, X, y):
        self.classes = np.unique(y)
        # For binary classification, transform to {-1, 1}
        y_transformed = np.where(y == self.classes[1], 1, -1)
        
        # Initialize predictions with zeros
        F = np.zeros(X.shape[0])
        
        # Iterate over estimators
        for _ in range(self.n_estimators):
            # Calculate negative gradient (residuals)
            p = 1.0 / (1.0 + np.exp(-2 * F))
            residuals = y_transformed - p
            
            # Fit a decision tree to the residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            
            # Update the model
            F += self.learning_rate * tree.predict(X)
            
            # Store the tree
            self.trees.append(tree)
        
        return self
    
    def predict_proba(self, X):
        # Calculate raw predictions
        F = np.zeros(X.shape[0])
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        
        # Transform to probabilities
        proba = 1.0 / (1.0 + np.exp(-2 * F))
        return np.vstack([1 - proba, proba]).T
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes[np.argmax(probas, axis=1)]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train and evaluate custom model
print("\nTraining custom Gradient Boosting model...")
custom_gb = CustomGradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)

# Measure training time
custom_start_time = time.time()
custom_gb.fit(X_train, y_train)
custom_train_time = time.time() - custom_start_time
print(f"Custom model training time: {custom_train_time:.4f} seconds")

# Predict and evaluate
custom_pred = custom_gb.predict(X_test)
custom_accuracy = accuracy_score(y_test, custom_pred)
print(f"Custom model accuracy: {custom_accuracy:.4f}")

# 5. Train and evaluate scikit-learn model
print("\nTraining scikit-learn Gradient Boosting model...")
sklearn_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# Measure training time
sklearn_start_time = time.time()
sklearn_gb.fit(X_train, y_train)
sklearn_train_time = time.time() - sklearn_start_time
print(f"Scikit-learn model training time: {sklearn_train_time:.4f} seconds")

# Predict and evaluate
sklearn_pred = sklearn_gb.predict(X_test)
sklearn_accuracy = accuracy_score(y_test, sklearn_pred)
print(f"Scikit-learn model accuracy: {sklearn_accuracy:.4f}")

# 6. Cross-validation for both models
print("\nPerforming cross-validation...")
# Custom model cross-validation
cv_scores_custom = []
cv_folds = 5
fold_size = len(X) // cv_folds
indices = np.arange(len(X))
np.random.shuffle(indices)

custom_cv_start_time = time.time()
for i in range(cv_folds):
    test_idx = indices[i*fold_size:(i+1)*fold_size]
    train_idx = np.concatenate([indices[:i*fold_size], indices[(i+1)*fold_size:]])
    X_train_cv, X_test_cv = X[train_idx], X[test_idx]
    y_train_cv, y_test_cv = y[train_idx], y[test_idx]
    
    custom_gb = CustomGradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
    custom_gb.fit(X_train_cv, y_train_cv)
    custom_pred_cv = custom_gb.predict(X_test_cv)
    cv_scores_custom.append(accuracy_score(y_test_cv, custom_pred_cv))
custom_cv_time = time.time() - custom_cv_start_time

# Scikit-learn model cross-validation
sklearn_cv_start_time = time.time()
cv_scores_sklearn = cross_val_score(sklearn_gb, X, y, cv=5)
sklearn_cv_time = time.time() - sklearn_cv_start_time

print(f"Custom model cross-validation accuracy: {np.mean(cv_scores_custom):.4f} (±{np.std(cv_scores_custom):.4f})")
print(f"Custom model cross-validation time: {custom_cv_time:.4f} seconds")
print(f"Scikit-learn model cross-validation accuracy: {np.mean(cv_scores_sklearn):.4f} (±{np.std(cv_scores_sklearn):.4f})")
print(f"Scikit-learn model cross-validation time: {sklearn_cv_time:.4f} seconds")

# 7. Compare results
print("\nComparison:")
print(f"Training time: Custom: {custom_train_time:.4f}s vs Scikit-learn: {sklearn_train_time:.4f}s")
print(f"Accuracy: Custom: {custom_accuracy:.4f} vs Scikit-learn: {sklearn_accuracy:.4f}")
print(f"Training time ratio (Custom/Scikit-learn): {custom_train_time/sklearn_train_time:.2f}")
print(f"Accuracy difference: {abs(custom_accuracy - sklearn_accuracy):.4f}")


Training custom Gradient Boosting model...
Custom model training time: 0.4251 seconds
Custom model accuracy: 0.9474

Training scikit-learn Gradient Boosting model...
Scikit-learn model training time: 0.4866 seconds
Scikit-learn model accuracy: 0.9561

Performing cross-validation...
Custom model cross-validation accuracy: 0.9345 (±0.0241)
Custom model cross-validation time: 2.1033 seconds
Scikit-learn model cross-validation accuracy: 0.9631 (±0.0210)
Scikit-learn model cross-validation time: 2.4059 seconds

Comparison:
Training time: Custom: 0.4251s vs Scikit-learn: 0.4866s
Accuracy: Custom: 0.9474 vs Scikit-learn: 0.9561
Training time ratio (Custom/Scikit-learn): 0.87
Accuracy difference: 0.0088
